In [ ]:
import duckdb
import pandas as pd
import numpy as np
import os
import time

import pyrootutils
root = pyrootutils.setup_root(
    search_from=os.path.abspath(''),
    indicator=[".git"],
    pythonpath=True, # add root directory to the PYTHONPATH (helps with imports)
    dotenv=True, # load environment variables from .env if exists in root directory
)
from utils.extractor import Extractor

# Cohort Identification

In [ ]:
datapath = 'PATH_TO_DATABASE' 
data_asset = 'DEID_CDW' # data_asset - Either DEID_CDW, DEID_OMOP, or PEDB

# Enter tables you are planning to query here
imagingfact             = duckdb.read_parquet(os.path.join(datapath,data_asset,'imagingfact','*.parquet'))
medicationorderfact     = duckdb.read_parquet(os.path.join(datapath,data_asset,'medicationorderfact','*.parquet'))
diagnosiseventfact      = duckdb.read_parquet(os.path.join(datapath,data_asset,'diagnosiseventfact','*.parquet'))
diagnosisterminologydim = duckdb.read_parquet(os.path.join(datapath,data_asset,'diagnosisterminologydim','*.parquet'))
procedureeventfact      = duckdb.read_parquet(os.path.join(datapath,data_asset,'procedureeventfact','*.parquet'))
patdurabledim           = duckdb.read_parquet(os.path.join(datapath,data_asset,'patdurabledim','*.parquet'))

#Set print counts flag
check_query_flag = True       #axilluary checks to see if query makes sense
PlumsExtractor = Extractor(num_results_flag = True, display_results_flag = True)

# Inclusion Criteria

In [ ]:
diagnosisQuery = f'''
/*
Created By: Michelle Tong
Create Date: 9/29/2023
Description: Select patients with LBP
*/

SELECT DISTINCT 
  diagnosiskey, 
  diagnosisname, 
  type, 
  value
FROM diagnosisterminologydim
WHERE 
  (
    (
      LOWER(diagnosisname) LIKE '%back%pain%'
      AND 
      (
      LOWER(diagnosisname) LIKE '%lumb%'
      OR LOWER(diagnosisname) LIKE '%low%'
      )
    )
    OR
    (
      value LIKE 'M51.1'
      OR value LIKE 'M54.4'
      OR value LIKE 'M54.9'
    )
  )
  AND 
  (
    type LIKE 'ICD-10-CM'
    --OR type LIKE 'ICD-9-CM'
  )
ORDER BY type, value
'''

# Run query and update relevant keys
results_df_pd = PlumsExtractor.run_query(diagnosisQuery, runtime_flag=True)

LBP_key_workinglist = results_df_pd['diagnosiskey'].drop_duplicates().tolist()
print('number of diagnosis keys:',len(LBP_key_workinglist))

results_df_pd[0:10]

In [ ]:
diagnosisQuery = f'''
/*
Created By: Michelle Tong
Create Date: 9/29/2023
Description: Select patients with LBP
*/

SELECT DISTINCT 
  patientdurablekey,
  diagnosisname,
  startdatekey,
  startdatekeyvalue
  --MIN(startdatekeyvalue) AS startdatekeyvalue_firstlbp
FROM 
  diagnosiseventfact
  WHERE
    diagnosiskey IN {tuple(LBP_key_workinglist)}
    AND
    startdatekey > 0
  --GROUP BY
    --patientdurablekey
  
  ORDER BY
    startdatekey,
    patientdurablekey
'''

# Run query and update relevant keys
tmp_LBP_df = PlumsExtractor.run_query(diagnosisQuery, runtime_flag=True)

patientdurablekey_workinglist = tmp_LBP_df['patientdurablekey'].drop_duplicates()
print('total patientdurablekeys: ',len(patientdurablekey_workinglist)) # number of rows in df

tmp_LBP_df[0:10]

In [ ]:
# import matplotlib.pyplot as plt

# # Extract year (first 4 digits)
# start_year = df_dates_tmp['startdatekey'].astype(str).str[:4].astype(int)

# # Plot histogram
# #start_year.value_counts().sort_index().plot(kind='bar', figsize=(10, 6), rot=0)
# plt.figure(figsize=(10, 6))
# plt.hist(start_year, bins=range(1980, 2024 + 1), edgecolor='black', align='left')

# plt.title('Count of Records by Year')
# plt.xlabel('Year')
# plt.ylabel('Count')
# plt.show()

# Exclusion Criteria

In [ ]:
bday_query = f"""
        SELECT 
          i.patientdurablekey, 
          i.startdatekey, 
          p.birthdate,
          DATE_ADD(p.birthdate, INTERVAL 18 YEAR) AS date_18_years
        FROM tmp_LBP_df as i
        JOIN patdurabledim as p
        ON 
          i.patientdurablekey = p.patientdurablekey
        WHERE 
          i.startdatekeyvalue < DATE_ADD(p.birthdate, INTERVAL 18 YEAR)
        ORDER BY
            birthdate DESC
"""

results_df = PlumsExtractor.run_query(bday_query, runtime_flag=True)

patientdurablekey_bday_excludelist = results_df['patientdurablekey'].drop_duplicates()
print('total patientdurablekeys: ',len(patientdurablekey_bday_excludelist)) # number of rows in df

results_df.head()

In [ ]:
# Exclude Surgery and Injections
# Procedures
surgery_list = ['DISCECTOMY','LAMINECTOMY','LAMINOTOMY','FORAMINOTOMY','FACETECTOMY']
surgery_abbrev_list = ['ALIF','ASF','PSF','OLIF','TLIF','XLIF']

surgery_query_txt = ' OR '.join(['procedurename LIKE \'%'+x+'%\'' for x in surgery_list])
surgery_abbrev_query_txt = ' OR '.join(['UPPER(procedurename) LIKE \'%'+x+'%\'' for x in surgery_abbrev_list])

surgeryQuery = f'''
/*
Created By: Michelle Tong
Create Date: 6/27/2023
Description: Select relevant info (fusion or surgeries for spine or interbody) and keys from procedure event fact table for study cohort.
Select relevant injections (injections, facet and nerve blocks, radiofrequency ablation) and keys from procedure event fact table for study cohort
*/

SELECT DISTINCT
  patientdurablekey,
  procedurekey,
  procedurename, 
  procedurecode,
  procedurecategory,
  procedurestartdatekey,
  procedureenddatekey,
  CASE
      WHEN UPPER(procedurename) LIKE '%INJECTION%' THEN 'INJECTION'
      WHEN UPPER(procedurename) LIKE '%BLOCK%' THEN 'BLOCK'
      WHEN UPPER(procedurename) LIKE '%ABLATION%' THEN 'RADIOFREQUENCY ABLATION'
      WHEN UPPER(procedurename) LIKE '%ARTHRODESIS%' THEN 'FUSION'
      WHEN UPPER(procedurename) LIKE '%FUSION%' THEN 'FUSION'
      WHEN UPPER(procedurename) LIKE '%ALIF%' THEN 'FUSION'
      WHEN UPPER(procedurename) LIKE '%ASF%' THEN 'FUSION'
      WHEN UPPER(procedurename) LIKE '%PSF%' THEN 'FUSION'
      WHEN UPPER(procedurename) LIKE '%OLIF%' THEN 'FUSION'
      WHEN UPPER(procedurename) LIKE '%TLIF%' THEN 'FUSION'
      WHEN UPPER(procedurename) LIKE '%XLIF%' THEN 'FUSION'
      WHEN UPPER(procedurename) LIKE '%SURGERY%' THEN 'SURGERY'
      WHEN UPPER(procedurename) LIKE '%OSTEOTOMY%' THEN 'OSTEOTOMY'
      ELSE 'SURGERY'
      END AS proceduretype
FROM 
  procedureeventfact
  WHERE 
    patientdurablekey IN {tuple(patientdurablekey_workinglist)}
    -- AND procedurestartdatekey > 0 -- 5/24/2024 excluding this line, difference of ~2400 patients
    AND
    (
        --FOR surgery (specific names)
        {surgery_query_txt}
        OR 
        --FOR fusions (full name)
        (
            (
            UPPER(procedurename) LIKE '%FUSION%'
            OR UPPER(procedurename) LIKE '%ARTHRODESIS%'
            )
            AND
            (
            UPPER(procedurename) LIKE '%SPINE%'
            OR UPPER(procedurename) LIKE '%INTERBODY%'
            OR UPPER(procedurename) LIKE '%LUMB%' -- LUMBAR
            )
        )
        OR
        --FOR fusions (abbreviations)
        (
            (
            {surgery_abbrev_query_txt}
            )
            AND
            (
            UPPER(procedurename) LIKE '%SPINE%'
            OR UPPER(procedurename) LIKE '%INTERBODY%'
            OR UPPER(procedurename) LIKE '%LUMBAR%' -- LUMBAR
            OR UPPER(procedurename) LIKE '%FUSION%'
            )
        )
        OR
        --FOR surgeries
        (
            (
            UPPER(procedurename) LIKE '%SURGERY%'
            OR UPPER(procedurename) LIKE '%OSTEOTOMY%'
            -- OR UPPER(procedurename) LIKE '%MIS%' -- currently this describes TLIF which is explicitly queried elsewhere
            )
            AND
            (
            UPPER(procedurename) LIKE '%SPINE%'
            OR UPPER(procedurename) LIKE '%DISC %' -- not DISCharge, lumbar/cervical/thoracic disc surgery
            -- OR UPPER(procedurename) LIKE '%CORD%' -- not contributing to HUE search
            OR UPPER(procedurename) like '%LUMB%' -- LUMBAR
            )
        )
        OR
        -- FOR injection
        (
            (
            UPPER(procedurename) LIKE '%INJECTION%' -- includes epidural injections
            OR UPPER(procedurename) LIKE '%BLOCK%'
            OR  (UPPER(procedurename) LIKE '%ABLATION%'
                AND UPPER(procedurename) LIKE '%RADIOFREQUENCY%'
                )
            )
            AND 
            (
            UPPER(procedurename) LIKE '%LUMB%'
              --OR 
              --(
                --UPPER(procedurename) LIKE '%SPINE%' -- adds 2: OTHER TYPES OF SPINE INJECTION: (SPECIFY), PERCUTANEOUS SPINE INJECTION POST KYPHOPLASTY, WITH FLURO GUIDANCE
                --AND UPPER(procedurename) NOT LIKE '%CERV%'
                --AND UPPER(procedurename) NOT LIKE '%THOR%'
                -- NERVE has many uses so will need to specify further
                -- CORD is only used for vocal cord
              --)
            )
        )
    )
ORDER BY
    patientdurablekey,
    procedurestartdatekey
    
'''

# Run query and update relevant keys
results_df = PlumsExtractor.run_query(surgeryQuery, runtime_flag=True)

patientdurablekey_procedure_excludelist = results_df['patientdurablekey'].drop_duplicates()
print('total patientdurablekeys: ',len(patientdurablekey_procedure_excludelist)) # number of rows in df

# #Check whether query makes sense
# if check_query_flag==True:
#     PlumsExtractor.col_to_list(results_df,'patientdurablekey')
#     PlumsExtractor.col_to_list(results_df,'encounterkey')
#     PlumsExtractor.col_to_list(results_df,'procedurename')
    
results_df.head()

In [ ]:
# Exclude diagnoses known to cause or confound LBP
# osteoarthritis (TBD)
#infection in bone, intervertebral disc, vertebral endplate and vertebral body, disc and adjacent vertebral body, disc and adjacent vertebral body
exclude_diag_list = ['osteomyelitis','discitis','spondylitis','spondylodiscitis','septic facet joint'] 
exclude_diag_query_txt = ' OR '.join(['LOWER(diagnosisname) LIKE \'%'+x+'%\'' for x in exclude_diag_list])

exclude_diagnosisQuery = f'''
/*
Created By: Michelle Tong
Create Date: 7/11/2023
Description: Select relevant patient, visit, and diagnosis keys from terminology table for patients who have radiculopath, stenosis, herniated discs
*/

SELECT 
  patientdurablekey, 
  diagnosiseventkey, 
  diagnosiskey, 
  diagnosisname, 
  encounterkey, 
  startdatekey, 
  enddatekey,
  CASE
  WHEN LOWER(diagnosisname) LIKE '%fracture%' THEN 'fracture'
  WHEN LOWER(diagnosisname) LIKE '%failed back%' THEN 'failed back'
  WHEN LOWER(diagnosisname) LIKE '%abscess%' THEN 'abscess'
  WHEN LOWER(diagnosisname) LIKE '%lymph%' THEN 'lymph'
  WHEN LOWER(diagnosisname) LIKE '%stroke%' THEN 'stroke'
  WHEN LOWER(diagnosisname) LIKE '%sepsis%' THEN 'sepsis'
  WHEN LOWER(diagnosisname) LIKE '%paresis%' THEN 'paresis'
  WHEN ({exclude_diag_query_txt}) THEN 'infection'
  WHEN (LOWER(diagnosisname) LIKE '%tumor%' OR LOWER(diagnosisname) LIKE '%metastasis%' OR LOWER(diagnosisname) LIKE '%oncology%') THEN 'cancer'
  ELSE 'category not specified'
  END AS diagnosistype
FROM 
  diagnosiseventfact
  
  WHERE
  patientdurablekey IN {tuple(patientdurablekey_workinglist)}
  AND
  (
    -- diagnosis exclusions
    {exclude_diag_query_txt}
    
    OR LOWER(diagnosisname) LIKE '%fracture%'
    OR LOWER(diagnosisname) LIKE '%failed back%' 
    OR LOWER(diagnosisname) LIKE '%abscess%' 
    
    OR LOWER(diagnosisname) LIKE '%tumor%'
    OR LOWER(diagnosisname) LIKE '%metastasis%'
    OR LOWER(diagnosisname) LIKE '%oncology%'
    OR LOWER(diagnosisname) LIKE '%lymph%'
    OR LOWER(diagnosisname) LIKE '%stroke%'
    OR LOWER(diagnosisname) LIKE '%sepsis%'
    OR LOWER(diagnosisname) LIKE '%paresis%'
    OR LOWER(diagnosisname) LIKE '%spondyloarthr%' 
  )
  
  ORDER BY
    diagnosiskey,
    patientdurablekey
'''


# Run query and update relevant keys
results_df = PlumsExtractor.run_query(exclude_diagnosisQuery, runtime_flag=True)

patientdurablekey_diagnosis_excludelist = results_df['patientdurablekey'].drop_duplicates()
print('total patientdurablekeys: ',len(patientdurablekey_diagnosis_excludelist)) # number of rows in df

#Check whether query makes sense
# if check_query_flag==True:
#     PlumsExtractor.col_to_list('encounterkey')
    
results_df.head()

In [ ]:
# Exclude diagnoses known to cause or confound LBP

exclude_diagnosisQuery = f'''
/*
Created By: Michelle Tong
Create Date: 5/24/2024
Description: Select relevant patient, visit, and diagnosis keys from terminology table for patients who have radiculopath, stenosis, herniated discs
*/

SELECT 
  patientdurablekey, 
  diagnosiseventkey, 
  diagnosiskey, 
  diagnosisname, 
  encounterkey, 
  startdatekey, 
  enddatekey,
  CASE
  WHEN LOWER(diagnosisname) LIKE '%infection%' THEN 'infection'
  WHEN LOWER(diagnosisname) LIKE '%trama%' THEN 'trama'
  ELSE 'category not specified'
  END AS diagnosistype
FROM 
  diagnosiseventfact
  
  WHERE
  patientdurablekey IN {tuple(patientdurablekey_workinglist)}
  AND
  (
    -- diagnosis exclusions
    LOWER(diagnosisname) LIKE '%infection%'
    OR LOWER(diagnosisname) LIKE '%trama%'
  )
  
  ORDER BY
    diagnosiskey,
    patientdurablekey
'''


# Run query and update relevant keys
results_df = PlumsExtractor.run_query(exclude_diagnosisQuery, runtime_flag=True)

patientdurablekey_infection_trama_excludelist = results_df['patientdurablekey'].drop_duplicates()
print('total patientdurablekeys: ',len(patientdurablekey_infection_trama_excludelist)) # number of rows in df

# #Check whether query makes sense
# if check_query_flag==True:
#     PlumsExtractor.col_to_list('encounterkey')
    
results_df.head()

# SELECT PATIENTS

In [ ]:
#INCLUSION
print('n=', len(patientdurablekey_workinglist))
print('unique n=', len(set(patientdurablekey_workinglist)))

In [ ]:
#EXCLUSION
print('exclude bday: ', len(set(patientdurablekey_bday_excludelist)))
print('exclude procedure: ', len(set(patientdurablekey_procedure_excludelist)))
print('exclude diag 1: ', len(set(patientdurablekey_diagnosis_excludelist)))
print('exclude diag 2: ', len(set(patientdurablekey_infection_trama_excludelist)))

print('exclude all diag: ', len(set(patientdurablekey_diagnosis_excludelist.to_list()+patientdurablekey_infection_trama_excludelist.to_list())))

In [ ]:
#COMBINE EXCLUSION
PID_surg_inj = patientdurablekey_procedure_excludelist.to_list()
print(len(set(PID_surg_inj)))

PID_surg_inj_diag = PID_surg_inj + patientdurablekey_diagnosis_excludelist.to_list()
print(len(set(PID_surg_inj_diag)))

PID_surg_inj_diag_infect_trama = PID_surg_inj_diag + patientdurablekey_infection_trama_excludelist.to_list()
print(len(set(PID_surg_inj_diag_infect_trama)))

PID_surg_inj_diag_infect_trama_over18yrs = PID_surg_inj_diag_infect_trama + patientdurablekey_bday_excludelist.to_list()
print(len(set(PID_surg_inj_diag_infect_trama_over18yrs)))

In [ ]:
#SELECTION
print('eligible n=', len(patientdurablekey_workinglist) - len(set(PID_surg_inj_diag_infect_trama_over18yrs)))
patientdurablekey_selection = set(patientdurablekey_workinglist)-set(PID_surg_inj_diag_infect_trama_over18yrs)
df_pdk = pd.DataFrame({'patientdurablekey': list(patientdurablekey_selection)})

In [ ]:
# SAVE OUTPUT
save_path = 'project_patient_list.csv'
os.makedirs(os.path.dirname(save_path), exist_ok=True) 
df_pdk.to_csv(save_path, index=False)